# Practica M30: Manipulación de datos con Pandas
#### En este Notebook se a leer el archivo fifa_eda.csv, revisar sus datos y practicar algunas operaciones de inserción, edición, ordenamiento y agrupación.

## Paso 1. Importar las librerías
#### Utilizo pandas para trabajar con el CSV, datetime para conocer el año actual y matplotlib para el gráfico opcional.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

## Paso 2. Cambiar el directorio y leer el CSV
#### Los archivos están en la carpeta Downloads de mi Mac. El código acepta tanto fifa_eda.csv como fifa_eda(1).csv, por si el archivo se descargó más de una vez.

In [ ]:
# Cambio a mi carpeta de Descargas
os.chdir("/Users/santiago/Downloads")
print("Directorio actual:", os.getcwd())

# Busco el archivo CSV
if os.path.exists("fifa_eda.csv"):
    archivo = "fifa_eda.csv"
elif os.path.exists("fifa_eda(1).csv"):
    archivo = "fifa_eda(1).csv"
else:
    raise FileNotFoundError("No encontré fifa_eda.csv en Downloads")

# Leo el archivo usando Pandas
fifa = pd.read_csv(archivo)
print("Archivo leído:", archivo)
print("Total de filas:", len(fifa))
print("Total de columnas:", len(fifa.columns))

## Paso 3. Mostrar las primeras 20 filas, las últimas 5 y 10 aleatorias
#### head() muestra las filas iniciales, tail() las finales y sample() selecciona filas aleatorias.

In [ ]:
print("PRIMERAS 20 FILAS")
display(fifa.head(20))

In [ ]:
print("ÚLTIMAS 5 FILAS")
display(fifa.tail(5))

In [ ]:
print("MUESTRA DE 10 JUGADORES")
display(fifa.sample(n=10, random_state=42))

## Paso 4. Información estadística y tipos de datos
#### Primero consulto los tipos de las columnas y después genero un resumen estadístico con describe().

In [ ]:
print("TIPOS DE DATOS")
print(fifa.dtypes)

print("INFORMACIÓN GENERAL")
fifa.info()

In [ ]:
print("ESTADÍSTICAS DEL DATASET")
display(fifa.describe())

## Paso 5. Convertir dos columnas a valores numéricos
#### Las columnas Value y Wage ya aparecen como numéricas en este CSV, pero utilizo pd.to_numeric() para practicar la conversión. Con errors="coerce", cualquier dato no convertible se transforma en valor faltante.

In [ ]:
# Convierto dos columnas a números
fifa["Value"] = pd.to_numeric(fifa["Value"], errors="coerce")
fifa["Wage"] = pd.to_numeric(fifa["Wage"], errors="coerce")

print("Tipos de datos después de convertir:")
print(fifa[["Value", "Wage"]].dtypes)

print("Estadísticas actualizadas:")
display(fifa.describe())

## Paso 6. Añadir la columna Years Playing
#### Resto el año de Joined al año actual para obtener los años transcurridos desde la incorporación registrada en el archivo. No significa necesariamente que el jugador siga en ese club.

In [ ]:
anio_actual = datetime.now().year
fifa["Years Playing"] = anio_actual - fifa["Joined"]

print("Año actual:", anio_actual)
display(fifa[["Name", "Club", "Joined", "Years Playing"]].head(15))

## Paso 7. Buscar todos los jugadores de México
#### Filtro la columna Nationality para quedarme solamente con los jugadores mexicanos. Muestro los resultados y los guardo para conservar la lista completa.

In [ ]:
jugadores_mexico = fifa[fifa["Nationality"] == "Mexico"]

print("Número de jugadores de México:", len(jugadores_mexico))

# Muestro todos los registros (no solamente las primeras filas)
with pd.option_context("display.max_rows", None):
    display(jugadores_mexico[["Name", "Club", "Overall", "Joined", "Years Playing"]])

# Guardo los jugadores mexicanos en un CSV
jugadores_mexico.to_csv("fifa_jugadores_mexico.csv", index=False)
print("Archivo creado: fifa_jugadores_mexico.csv")

## Paso 8. Ordenar los jugadores por Release Clause
#### Ordeno de mayor a menor con sort_values(). Muestro los 20 primeros y guardo el listado completo. Release Clause es la cláusula de rescisión, no el sueldo; el sueldo corresponde a Wage.

In [ ]:
fifa_ordenado = fifa.sort_values(by="Release Clause", ascending=False)

print("20 jugadores con mayor Release Clause:")
display(fifa_ordenado[["Name", "Nationality", "Club", "Release Clause"]].head(20))

# Guardo todo el dataset ya ordenado
fifa_ordenado.to_csv("fifa_ordenado_release_clause.csv", index=False)
print("Archivo creado: fifa_ordenado_release_clause.csv")

## Paso 9. Crear un nuevo dataset agrupado por el año Joined
#### Con groupby() cuento cuántos jugadores tienen cada año de incorporación. Después guardo la nueva tabla en CSV.

In [ ]:
jugadores_por_anio = fifa.groupby("Joined").size().reset_index(name="Numero de jugadores")

print("Jugadores agrupados por año de incorporación:")
display(jugadores_por_anio)

jugadores_por_anio.to_csv("fifa_jugadores_por_anio.csv", index=False)
print("Archivo creado: fifa_jugadores_por_anio.csv")

## Paso 10 (opcional). Gráfico del número de jugadores por año 
#### Creo un gráfico de barras sencillo a partir de la tabla agrupada.

In [ ]:
plt.figure(figsize=(11, 5))
plt.bar(jugadores_por_anio["Joined"], jugadores_por_anio["Numero de jugadores"])
plt.title("Jugadores por año de incorporación")
plt.xlabel("Año de incorporación (Joined)")
plt.ylabel("Número de jugadores")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("fifa_jugadores_por_anio.png")
plt.show()
print("Gráfico guardado: fifa_jugadores_por_anio.png")

## Paso 11. Prueba sencilla de edición y eliminación
#### La columna Years Playing fue un ejemplo de inserción. Ahora hago una prueba de edición y eliminación utilizando una copia de cinco registros, para no modificar los datos originales.

In [ ]:
# Hago una copia pequeña para practicar
prueba = fifa.head(5).copy()
print("ANTES DEL CAMBIO")
display(prueba[["Name", "Club"]])

# Edito el equipo del primer jugador en la copia
prueba.loc[prueba.index[0], "Club"] = "Club de ejemplo"

# Elimino la segunda fila solamente de la copia
prueba = prueba.drop(prueba.index[1])

print("DESPUÉS DE EDITAR Y ELIMINAR")
display(prueba[["Name", "Club"]])